# Step 4 — Build Analysis-Ready Dataset: Time Categories + Daylight Fraction

**Input**: `output/hamburg_bike_counts_hourly_5min_clean_v2.csv.gz` (output of Step 3)
**Output**:
- `output/processed_hourly_bike_counts_time_categories.csv.gz` (main table, hourly rows)
- `output/station_metadata.csv` (station attribute dimension table)

## Decisions made in this step (read this first before handover)

1. **The main table is normalized**: only `station_id` is kept as the foreign key; static station
   attributes (name, coordinates, service_name, etc.) are moved to `station_metadata.csv` and can
   be joined back on `station_id`. This keeps the main table smaller, for easier download and
   downstream processing.
2. **Coordinates**: sunrise/sunset are computed once for the whole city using a single
   representative point (lat=53.55, lon=10.00), not separately per station. Hamburg's east-west
   span is small, so the error from this simplification is usually within 1-2 minutes -- negligible.
3. **Timezone**: following the Step 3 decision, `datetime_hamburg` is used as-is as Hamburg local
   time, with no conversion. Sunrise/sunset are computed with the `astral` package using the
   `Europe/Berlin` timezone (which automatically handles the CET/CEST daylight-saving switch);
   the tzinfo is then stripped so it can be compared minute-by-minute directly against
   `datetime_hamburg` (naive).
   Note: if `hour_utc` later turns out to really be UTC (see the warning in the Step 3 notebook),
   `daylight_fraction` would be systematically off by about 2 hours in summer months and about 1
   hour in winter months. If that needs fixing later, only Step 3 needs a
   `tz_convert('Europe/Berlin')` added -- the logic in this step would not need to change.
4. **Hamburg public holidays**: computed with the Python `holidays` package
   (`holidays.Germany(subdiv='HH')`), replacing the two hand-maintained lists that used to live in
   separate notebooks (`3_Split_categories.ipynb` / `4_timeseries_check.ipynb`) and could drift out
   of sync. Checked and confirmed: within the analysis period (all of 2025 plus Jan-Feb 2026), all
   three lists match exactly.
5. **weekday_peak hour boundary (previously an open question, settled here)**:
   - `3_Split_categories.ipynb` used `[6,7,8,16,17]` (not including 9 or 18)
   - `4_timeseries_check.ipynb` used `6<=hr<=9` and `16<=hr<=18` (including 9 and 18)
   - Neither matches the official HVV definition (HVV's own definition is 6-10 and 15-19, and
     that's a public-transit standard, not a bicycle one, anyway)
   - **A data-driven choice was made instead**: checking the average hourly traffic by
     weekday hour, 9am (23.3) is already lower than 10am (25.8) -- so 9am doesn't behave like peak
     hour at all; but 6pm (24.1) is still 49% higher than 7pm (16.2), so 6pm is clearly still in
     the tail end of the evening peak. This notebook therefore uses **morning peak = 6, 7, 8
     (excluding 9), evening peak = 16, 17, 18 (including 18)** -- both peak windows are 3 hours.
   - This definition does not exactly match either of the two earlier notebooks. If downstream
     analysis is compared against older results, be sure to note that this new definition is in use.


In [1]:
import pandas as pd
import numpy as np
from astral import LocationInfo
from astral.sun import sun
import holidays

INPUT_PATH = "output/hamburg_bike_counts_hourly_5min_clean_v2.csv.gz"
OUTPUT_MAIN_PATH = "output/processed_hourly_bike_counts_time_categories.csv.gz"
OUTPUT_META_PATH = "output/station_metadata.csv"

HAMBURG_LAT = 53.55
HAMBURG_LON = 10.00
EXPECTED_RECORDS_FULL_PERIOD = 10176  # 424 days x 24 hours

# Data-driven peak-hour set (see note #5 above)
PEAK_HOURS = {6, 7, 8, 16, 17, 18}

In [2]:
# 1. Load the Step 3 output
df = pd.read_csv(
    INPUT_PATH,
    dtype={
        'station_name': 'category', 'datastream_id': 'int64', 'station_id': 'int64',
        'service_name': 'category', 'layer_name': 'category', 'resolution': 'category',
        'longitude_wgs84': 'float64', 'latitude_wgs84': 'float64',
        'bike_count_hourly': 'float64',
        'extreme_flag_relative_p995': 'bool', 'extreme_count_gt1000': 'bool', 'extreme_count_gt2000': 'bool',
        'readings_in_hour': 'int64', 'readings_expected': 'int64', 'coverage_pct': 'float64',
        'low_coverage': 'bool', 'quality_flag': 'category',
    },
    parse_dates=['datetime_hamburg'],
)
print(f"Loaded: {len(df):,} rows, {df['station_id'].nunique()} stations")

Loaded: 3,227,633 rows, 328 stations


In [3]:
# 2. Build station_metadata.csv (static station attributes + completeness)
station_meta = df.groupby('station_id', observed=True).agg(
    station_name=('station_name', 'first'),
    datastream_id=('datastream_id', 'first'),
    service_name=('service_name', 'first'),
    layer_name=('layer_name', 'first'),
    resolution=('resolution', 'first'),
    longitude_wgs84=('longitude_wgs84', 'first'),
    latitude_wgs84=('latitude_wgs84', 'first'),
    n_observed_records=('bike_count_hourly', 'size'),
).reset_index()

station_meta['n_expected_records'] = EXPECTED_RECORDS_FULL_PERIOD
station_meta['is_complete_full_period'] = station_meta['n_observed_records'] == EXPECTED_RECORDS_FULL_PERIOD

station_meta = station_meta[['station_id', 'station_name', 'datastream_id', 'service_name', 'layer_name',
                              'resolution', 'longitude_wgs84', 'latitude_wgs84',
                              'n_observed_records', 'n_expected_records', 'is_complete_full_period']]
station_meta.to_csv(OUTPUT_META_PATH, index=False)
print(f"station_metadata.csv done: {len(station_meta)} stations, "
      f"fully-covered stations: {station_meta['is_complete_full_period'].sum()}")

station_metadata.csv done: 328 stations, fully-covered stations: 241


In [4]:
# 3. Main table keeps only station_id as the foreign key; other station attribute columns are
#    dropped here (already saved into station_metadata.csv)
main_cols = ['station_id', 'hour_utc', 'datetime_hamburg', 'date', 'year', 'month', 'hour_24',
             'bike_count_hourly',
             'extreme_flag_relative_p995', 'extreme_count_gt1000', 'extreme_count_gt2000',
             'readings_in_hour', 'readings_expected', 'coverage_pct', 'low_coverage', 'quality_flag']
main = df[main_cols].copy()
del df

In [5]:
# 4. Compute Hamburg daily sunrise/sunset with astral (Europe/Berlin, DST handled automatically)
loc = LocationInfo(name='Hamburg', region='Germany', timezone='Europe/Berlin',
                    latitude=HAMBURG_LAT, longitude=HAMBURG_LON)
berlin_tz = loc.tzinfo

main['date_parsed'] = pd.to_datetime(main['date']).dt.date
unique_dates = sorted(main['date_parsed'].unique())

sun_records = []
for d in unique_dates:
    s = sun(loc.observer, date=d, tzinfo=berlin_tz)
    # Strip tzinfo so this can be compared against datetime_hamburg as the same kind of
    # "local time number"
    sun_records.append((d, s['sunrise'].replace(tzinfo=None), s['sunset'].replace(tzinfo=None)))

sun_df = pd.DataFrame(sun_records, columns=['date_parsed', 'sunrise', 'sunset'])
main = main.merge(sun_df, on='date_parsed', how='left')
print(f"Sunrise/sunset computed for {len(sun_df)} days")

Sunrise/sunset computed for 424 days


In [6]:
# 5. daylight_fraction: overlap in minutes between the hour and [sunrise, sunset), divided by 60
#    (vectorized)
hour_start = main['datetime_hamburg']
hour_end = main['datetime_hamburg'] + pd.Timedelta(hours=1)

overlap_start = np.maximum(hour_start.values.astype('datetime64[ns]'),
                            main['sunrise'].values.astype('datetime64[ns]'))
overlap_end = np.minimum(hour_end.values.astype('datetime64[ns]'),
                          main['sunset'].values.astype('datetime64[ns]'))

overlap_minutes = (overlap_end - overlap_start) / np.timedelta64(1, 'm')
overlap_minutes = np.clip(overlap_minutes, 0, 60)
main['daylight_fraction'] = overlap_minutes / 60.0

# 6. Day/night weighted counts (the original bike_count_hourly is never overwritten)
main['day_count_weighted'] = main['bike_count_hourly'] * main['daylight_fraction']
main['night_count_weighted'] = main['bike_count_hourly'] * (1 - main['daylight_fraction'])

In [7]:
# 7. weekday / weekend
main['day_of_week'] = main['datetime_hamburg'].dt.dayofweek  # Monday=0
main['is_weekend'] = main['day_of_week'] >= 5
main['day_type'] = np.where(main['is_weekend'], 'weekend', 'weekday')

# 8. Hamburg public holidays (holidays package, Hamburg subdivision -- replaces the two
#    hand-maintained lists that used to live in separate notebooks)
hh_holidays = holidays.Germany(subdiv='HH', years=[2025, 2026])
holiday_dates = set(hh_holidays.keys())
main['is_holiday'] = main['date_parsed'].isin(holiday_dates)

In [8]:
# 9. time_period classification (peak-hour definition explained in note #5 at the top)
hour_int = main['hour_24'].astype(int)
is_peak_hour = hour_int.isin(PEAK_HOURS)
is_ph_day = main['is_weekend'] | main['is_holiday']

main['is_hvv_peak_hour'] = is_peak_hour

conditions = [is_ph_day, (~is_ph_day) & is_peak_hour, (~is_ph_day) & (~is_peak_hour)]
choices = ['weekend_public_holiday', 'weekday_peak', 'weekday_offpeak']
main['time_period'] = np.select(conditions, choices, default='weekday_offpeak')

print(main['time_period'].value_counts())

time_period
weekday_offpeak           1666956
weekend_public_holiday    1005040
weekday_peak               555637
Name: count, dtype: int64


In [9]:
# 10. Reorder columns and write out
final_cols = ['station_id', 'hour_utc', 'datetime_hamburg', 'date', 'year', 'month', 'hour_24', 'day_of_week',
              'is_weekend', 'day_type', 'is_holiday', 'is_hvv_peak_hour', 'time_period',
              'bike_count_hourly', 'daylight_fraction', 'day_count_weighted', 'night_count_weighted',
              'extreme_flag_relative_p995', 'extreme_count_gt1000', 'extreme_count_gt2000',
              'readings_in_hour', 'readings_expected', 'coverage_pct', 'low_coverage', 'quality_flag']
main = main[final_cols]

main.to_csv(OUTPUT_MAIN_PATH, index=False, compression='gzip')
print(f"Written: {OUTPUT_MAIN_PATH}")
print(f"Row count: {len(main):,}")

Written: output/processed_hourly_bike_counts_time_categories.csv.gz
Row count: 3,227,633


## Processing summary & sanity checks

In [10]:
# Summary: rows and total count per time_period
summary = main.groupby('time_period').agg(
    rows=('bike_count_hourly', 'size'),
    total_count=('bike_count_hourly', 'sum'),
    mean_count=('bike_count_hourly', 'mean'),
    stations=('station_id', 'nunique')
).reset_index()
print(summary.to_string(index=False))
print(f"\nTotal rows: {len(main):,}  Total count: {main['bike_count_hourly'].sum():,.0f}")

           time_period    rows  total_count  mean_count  stations
       weekday_offpeak 1666956   30839829.0   18.500686       328
          weekday_peak  555637   21342542.0   38.410945       328
weekend_public_holiday 1005040   13147099.0   13.081170       328

Total rows: 3,227,633  Total count: 65,329,470


In [11]:
# daylight_fraction sanity check
n_zero = (main['daylight_fraction'] == 0).sum()
n_one = (main['daylight_fraction'] == 1).sum()
n_between = ((main['daylight_fraction'] > 0) & (main['daylight_fraction'] < 1)).sum()

print(f"daylight_fraction == 0 (fully night): {n_zero:,} rows ({n_zero/len(main)*100:.2f}%)")
print(f"daylight_fraction == 1 (fully day): {n_one:,} rows ({n_one/len(main)*100:.2f}%)")
print(f"daylight_fraction between 0 and 1 (crosses sunrise/sunset): {n_between:,} rows ({n_between/len(main)*100:.2f}%)")

# Theoretical value: days x 2 (sunrise+sunset) x stations; actual will be slightly lower because
# 87 stations don't have full coverage
n_days = len(unique_dates)
n_stations = main['station_id'].nunique()
print(f"\nApprox. theoretical crossing rows = {n_days} days x 2 x {n_stations} stations = {n_days*2*n_stations:,}")
print(f"Actual crossing rows = {n_between:,}")

daylight_fraction == 0 (fully night): 1,501,587 rows (46.52%)
daylight_fraction == 1 (fully day): 1,457,086 rows (45.14%)
daylight_fraction between 0 and 1 (crosses sunrise/sunset): 268,960 rows (8.33%)

Approx. theoretical crossing rows = 424 days x 2 x 328 stations = 278,144
Actual crossing rows = 268,960
